# Tutorial on using the `mpl_pipeline.py` module

This tutorial will guide you in utiling the `mlp_pipeline.py` module to its fullest capability by importing the Fedarlist papers and performing an authroship attribution experiment utilizing a Multi Layer Perceptron classfier. It trains on known papers (**HAMILTON**, **MADISON**), evaluates holdout and cross-validation performance, and predicts likely authorship for unknown papers (**DISPUTED**, **COAUTHORED**).

### Detecting GPU
This cell detects if your machine has a GPU available for usage, allowing the notebook to run the rest of the experiment in your GPU if applicable*

*NOTE FOR DEVELOPERS: the notebook still runs on CPU regardless, a new library must be chosen and the code refactored if GPU is to be used

In [1]:
import importlib.util
import subprocess


def detect_gpu() -> dict:
    """Detect whether a GPU is visible and whether a Python GPU backend is usable."""
    info = {
        "gpu_visible": False,
        "backend": None,
        "details": "No GPU detected.",
    }

    if importlib.util.find_spec("torch") is not None:
        import torch

        if torch.cuda.is_available():
            info["gpu_visible"] = True
            info["backend"] = "torch"
            info["details"] = (
                f"PyTorch CUDA available with {torch.cuda.device_count()} device(s): "
                + ", ".join(
                    torch.cuda.get_device_name(i)
                    for i in range(torch.cuda.device_count())
                )
            )
            return info

    if importlib.util.find_spec("tensorflow") is not None:
        import tensorflow as tf

        gpus = tf.config.list_physical_devices("GPU")
        if gpus:
            info["gpu_visible"] = True
            info["backend"] = "tensorflow"
            info["details"] = f"TensorFlow detected {len(gpus)} GPU device(s)."
            return info

    try:
        probe = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            capture_output=True,
            text=True,
            check=False,
        )
        gpu_names = [line.strip() for line in probe.stdout.splitlines() if line.strip()]
        if probe.returncode == 0 and gpu_names:
            info["gpu_visible"] = True
            info["backend"] = "nvidia-smi"
            info["details"] = "NVIDIA GPU visible: " + ", ".join(gpu_names)
    except FileNotFoundError:
        pass

    return info


gpu_status = detect_gpu()
USE_GPU = gpu_status["gpu_visible"] and gpu_status["backend"] in {"torch", "tensorflow"}

print(gpu_status["details"])
print(f"USE_GPU={USE_GPU}")

if gpu_status["gpu_visible"]:
    print(
        "Note: The current `mlp_pipeline` uses scikit-learn MLP, which trains on CPU. "
        "GPU detection is exposed for environment checks and future GPU-enabled model paths."
    )

No GPU detected.
USE_GPU=False


## Imports

First me must import all necessary libraries and Lexos components

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from lexos.classification.mlp_pipeline import (
    MLPPipelineConfig,
    run_mlp_authorship_pipeline,
    save_mlp_unknown_predictions,
)

## Locate Data and Build Datasets

We locate the `fed_papers` directory, separate known-author papers for training, and collect unknown papers for inference.

In [ ]:
SEED = 42
np.random.seed(SEED)

base = Path.cwd()
search_roots = [base] + list(base.parents)

data_dir = next(
    (root / "fed_papers" for root in search_roots if (root / "fed_papers").exists()),
    None,
)
if data_dir is None:
    raise FileNotFoundError(
        "Could not locate 'fed_papers' from the current notebook location."
    )

train_dirs = ["HAMILTON", "MADISON"]
unknown_dirs = ["DISPUTED", "COAUTHORED"]

# Collect all known-author texts and labels for training
train_files = []
train_labels = []
for author in train_dirs:
    files = sorted((data_dir / author).glob("*.txt"))
    train_files.extend(files)
    train_labels.extend([author] * len(files))

unknown_files = []
unknown_sets = []
for subset in unknown_dirs:
    files = sorted((data_dir / subset).glob("*.txt"))
    unknown_files.extend(files)
    unknown_sets.extend([subset] * len(files))

train_texts = [p.read_text(encoding="utf-8", errors="ignore") for p in train_files]
unknown_texts = [p.read_text(encoding="utf-8", errors="ignore") for p in unknown_files]
unknown_ids = [p.name for p in unknown_files]

print("Using data directory:", data_dir)
print(f"Training docs: {len(train_texts)}")
print(f"Unknown docs: {len(unknown_texts)}")
print(pd.Series(train_labels).value_counts())

Using data directory: /home/mango/Lexos_Independant_Research/lexos/doc_src/docs/tutorials/classification/fed_papers
Training docs: 65
Unknown docs: 15
HAMILTON    51
MADISON     14
Name: count, dtype: int64


## The `MLPPipelineConfig` Class

The pipeline performs leakage-safe preprocessing, holdout evaluation, cross-validation, final refit, and optional inference on unknown documents.

### Properties

- `seed`: int = 42
- `min_df`: int = 2
- `test_size`: float = 0.2
- `cv_splits`: int = 5
- `include_bigrams`: bool = True
- `use_corpus_stats_features`: bool = False
- `corpus_stats_feature_columns`: tuple[str, ...] | None = None
- `use_smote`: bool = True
- `mlp_kwargs`: dict[str, Any] with default values:
- `hidden_layer_sizes`=(64,), activation=relu, solver=adam, alpha=1e-4, learning_rate_init=1e-3, max_iter=1000



In [ ]:
if USE_GPU:
    print(
        "GPU detected; current mlp_pipeline path remains CPU-bound (scikit-learn MLP backend)."
    )
else:
    print("No compatible GPU backend detected. Running pipeline on CPU.")

cfg = MLPPipelineConfig(
    seed=SEED,
    min_df=2,
    test_size=0.2,
    cv_splits=5,
    include_bigrams=True,
    use_smote=True,
    use_corpus_stats_features=True,
    corpus_stats_feature_columns=None,
    mlp_kwargs={
        "hidden_layer_sizes": (64,),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-4,
        "learning_rate_init": 1e-3,
        "max_iter": 1000,
    },
)

No compatible GPU backend detected. Running pipeline on CPU.


## Running the pipeline

In [ ]:
results = run_mlp_authorship_pipeline(
    train_data=train_texts,
    train_labels=train_labels,
    test_data=unknown_texts,
    test_ids=unknown_ids,
    config=cfg,
)

## Holdout Evaluation

Metrics below come from the leakage-safe holdout split generated inside the pipeline.

In [5]:
print("Holdout metrics:")
print(results.holdout_metrics)

print("\nClassification report:")
results.holdout_report

Holdout metrics:
{'accuracy': 0.7692307692307693, 'balanced_accuracy': 0.5, 'macro_f1': 0.43478260869565216}

Classification report:


,precision,recall,f1-score,support
HAMILTON,0.769231,1.000000,0.869565,10.000000
MADISON,0.000000,0.000000,0.000000,3.000000
accuracy,0.769231,0.769231,0.769231,0.769231
macro avg,0.384615,0.500000,0.434783,13.000000
weighted avg,0.591716,0.769231,0.668896,13.000000


In [6]:
print("Holdout confusion matrix:")
results.holdout_confusion_matrix

Holdout confusion matrix:


,pred_HAMILTON,pred_MADISON
true_HAMILTON,10,0
true_MADISON,3,0


## Cross-Validation Metrics

These values summarize the leakage-safe stratified CV executed inside the pipeline.

In [7]:
print("Per-fold CV metrics:")
results.cv_fold_metrics

print("\nMean CV metrics:")
results.cv_mean_metrics

Per-fold CV metrics:

Mean CV metrics:


{'accuracy': 0.876923076923077,
 'balanced_accuracy': 0.7233333333333333,
 'macro_f1': 0.7095505067015364}

## Why We Use Holdout Evaluation and Cross-Validation

### Holdout Evaluation
Holdout evaluation measures model performance on a subset of labeled data that was **not used during training** (the test split).  
This gives a quick estimate of how well the pipeline generalizes to unseen documents.

In this notebook, holdout outputs include:
- **Aggregate metrics** (for example, accuracy, precision, recall, F1)
- **Classification report** (per-class performance)
- **Confusion matrix** (which labels are being mixed up)

**Why it is helpful:**  
It provides an easy-to-interpret “first check” of real-world behavior and highlights specific error patterns between Hamilton and Madison predictions.

### Cross-Validation
Cross-validation repeats training/evaluation across multiple stratified folds, each time changing which samples are held out.  
Instead of relying on one random split, it produces performance over several splits and reports both **per-fold** and **mean** metrics.

**Why it is helpful:**  
- Reduces dependence on a single train/test partition  
- Gives a more stable estimate of expected performance  
- Helps detect variance/instability (e.g., if fold scores fluctuate widely)

### How to Use Both Together
- Use **holdout metrics** for a direct, concrete snapshot of model performance.
- Use **cross-validation metrics** to judge robustness and reliability of that snapshot.

When both are consistent and strong, confidence in the model increases before interpreting disputed/coauthored-paper predictions.

## Inference on Unknown Papers

Predictions below are generated by the final model refit on all known labeled papers.

In [8]:
results_with_sets = results.test_predictions.merge(
    pd.DataFrame({"sample_id": unknown_ids, "set": unknown_sets}),
    on="sample_id",
    how="left",
)

results_with_sets = results_with_sets[
    ["sample_id", "set", "predicted_label"]
    + [col for col in results_with_sets.columns if col.startswith("p_")]
]

results_with_sets.sort_values(["set", "sample_id"]).reset_index(drop=True)

,sample_id,set,predicted_label,p_hamilton,p_madison
0,FED_18_C.txt,COAUTHORED,MADISON,4.720499e-04,0.999528
1,FED_19_C.txt,COAUTHORED,MADISON,5.305540e-02,0.946945
2,FED_20_C.txt,COAUTHORED,MADISON,8.161795e-03,0.991838
3,FED_49_D.txt,DISPUTED,MADISON,3.642817e-05,0.999964
4,FED_50_D.txt,DISPUTED,MADISON,1.314791e-01,0.868521
5,FED_51_D.txt,DISPUTED,MADISON,3.342683e-06,0.999997
6,FED_52_D.txt,DISPUTED,MADISON,3.965592e-05,0.999960
7,FED_53_D.txt,DISPUTED,MADISON,7.270753e-05,0.999927
8,FED_54_D.txt,DISPUTED,MADISON,4.114377e-06,0.999996
9,FED_55_D.txt,DISPUTED,MADISON,7.738502e-05,0.999923


## Save Predictions

Export unknown-paper predictions to CSV for downstream analysis.

In [9]:
output_csv = data_dir / "neural_authorship_predictions_pipeline.csv"
saved_path = save_mlp_unknown_predictions(results, output_csv)
print("Saved predictions:", saved_path)

Saved predictions: /home/mango/Lexos_Independant_Research/lexos/doc_src/docs/tutorials/classification/fed_papers/neural_authorship_predictions_pipeline.csv
